# MCP server 5 — Time-series evidence, catalogs, and recipes

Exercise the core TSFM workflow through the live MCP server: discover tasks, create and inspect time-series evidence, browse model and feature catalogs, inspect the recipe contract, optionally execute a lightweight forecast, and retrieve its provenance.

**Tutorial contract:** run cells from top to bottom. Evidence and catalog discovery run by default. Catalog-administration tools are discovered but never called. Lightweight recipe execution is opt-in because it writes run/result records and requires the optional ML runtime.


In [1]:
from pathlib import Path
import asyncio, json, os, sys
import numpy as np
import pandas as pd

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("python:", sys.version.split()[0])


repo: /Users/chathurangishyalika/IBM/AssetOpsBench
python: 3.12.13


In [ ]:
# Load environment variables from .env file in the repository root
from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None
    
repo = find_repo()

if repo:
    load_dotenv(repo / ".env", override=False)

## 1. Create the MCP client


In [2]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MCP_ENV = os.environ.copy()

async def tsfm_request(operation, tool_name=None, arguments=None):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), "tsfm-mcp-server"],
        cwd=str(REPO),
        env=MCP_ENV,
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if operation == "list":
                return await session.list_tools()
            return await session.call_tool(tool_name, arguments or {})

def parse_result(result):
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        payload = text
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(payload["error"])
    if isinstance(payload, str) and (
        payload.startswith("Unknown tool:") or payload.startswith("Error executing tool")
    ):
        raise RuntimeError(payload)
    return payload

async def list_tsfm_tools():
    response = await tsfm_request("list")
    return [
        {"name": tool.name, "description": tool.description, "schema": tool.inputSchema}
        for tool in response.tools
    ]

async def call_tsfm(name, **arguments):
    return parse_result(await tsfm_request("call", name, arguments))


## 2. Discover and validate the live contract

The server currently exposes a broad interface that includes evidence, discovery, execution, result-ledger, and catalog-administration tools. This tutorial validates the core workflow subset instead of executing every administrative mutation.


In [3]:
tools = await list_tsfm_tools()
contract = {
    tool["name"]: list(tool["schema"].get("properties", {}))
    for tool in tools
}
print("TSFM tools exposed:", len(contract))
contract


TSFM tools exposed: 41


{'list_tasks': [],
 'profile_series': ['dataset_path', 'timestamp_column', 'channels'],
 'characterize_series': ['dataset_path',
  'timestamp_column',
  'channels',
  'groups',
  'group_rules'],
 'data_quality': ['dataset_path', 'timestamp_column'],
 'list_features': ['kind', 'status'],
 'list_models': ['task_id', 'domain', 'status'],
 'search_models': ['text', 'tags', 'status'],
 'find_models': ['task_id',
  'min_context_length',
  'prediction_length',
  'domain',
  'top_k'],
 'describe_candidates': ['task_id', 'top_k', 'domain'],
 'describe_models': ['model_ids'],
 'count_models': [],
 'list_domains': ['task_id'],
 'get_model_lineage': ['model_id'],
 'register_model': ['model'],
 'model_template': [],
 'register_finetuned': ['model_id',
  'checkpoint_path',
  'base_model_id',
  'context_length',
  'prediction_length',
  'description',
  'domain'],
 'update_model': ['model_id', 'fields'],
 'deprecate_model': ['model_id', 'reason'],
 'new_model_version': ['model_id', 'fields', 'new_mod

In [4]:
CORE_TOOLS = {
    "list_tasks", "profile_series", "characterize_series", "data_quality",
    "list_models", "find_models", "describe_models", "count_models", "list_domains",
    "list_features", "count_features", "describe_features",
    "recipe_template", "run_recipe", "get_run", "list_runs",
    "get_result", "list_results",
}
CATALOG_MUTATIONS = {
    "register_model", "register_finetuned", "update_model", "deprecate_model",
    "new_model_version", "register_feature", "update_feature",
    "deprecate_feature", "new_feature_version",
}
missing_core = CORE_TOOLS - set(contract)
assert not missing_core, f"Required TSFM tools are missing: {sorted(missing_core)}"
print("Core tutorial contract is compatible.")
print("Catalog mutations discovered but intentionally not executed:", sorted(CATALOG_MUTATIONS & set(contract)))


Core tutorial contract is compatible.
Catalog mutations discovered but intentionally not executed: ['deprecate_feature', 'deprecate_model', 'new_feature_version', 'new_model_version', 'register_feature', 'register_finetuned', 'register_model', 'update_feature', 'update_model']


## 3. List standardized TSFM tasks

This call is lightweight and does not load a checkpoint or require CouchDB.


In [5]:
tasks = await call_tsfm("list_tasks")
assert len(tasks.get("tasks", [])) == 8
[
    {
        "task_id": task.get("task_id"),
        "description": task.get("description"),
        "metrics": task.get("metrics"),
    }
    for task in tasks["tasks"]
]


[{'task_id': 'tsfm_forecasting',
  'description': 'Predict future values of a series over a horizon.',
  'metrics': ['mase', 'wql', 'smape', 'mae']},
 {'task_id': 'tsfm_regression',
  'description': 'Predict a real-valued target from a series (e.g. remaining useful life).',
  'metrics': ['r2', 'mae', 'rmse']},
 {'task_id': 'tsfm_classification',
  'description': 'Assign a discrete class/label to a series (e.g. fault type).',
  'metrics': ['accuracy', 'f1', 'auroc']},
 {'task_id': 'tsfm_anomaly_detection',
  'description': 'Score/flag abnormal points or ranges in a series.',
  'metrics': ['auc_pr', 'range_f1']},
 {'task_id': 'tsfm_imputation',
  'description': 'Fill missing values in a series.',
  'metrics': ['mae', 'crps']},
 {'task_id': 'tsfm_evaluation',
  'description': 'Score predictions against ground truth / benchmark pipelines.',
  'metrics': ['mae', 'rmse']},
 {'task_id': 'tsfm_similarity_search',
  'description': 'Find the series most similar to a query (embedding + top-k retr

## 4. Create a reproducible tutorial time series

The synthetic CSV is an explicit file pointer for the TSFM evidence tools. A small gap is included so the data-quality tool has something meaningful to report.


In [6]:
DATASET = ARTIFACTS / "tsfm_tutorial_series.csv"
N_OBSERVATIONS = 120
index = np.arange(N_OBSERVATIONS)
frame = pd.DataFrame({
    "timestamp": pd.date_range("2026-01-01", periods=N_OBSERVATIONS, freq="h"),
    "temperature": 20.0 + np.sin(index / 8.0) + np.linspace(0.0, 2.0, N_OBSERVATIONS),
    "pressure": 100.0 + 0.5 * np.cos(index / 10.0),
})
frame.loc[40, "temperature"] = np.nan
frame.to_csv(DATASET, index=False)
print("dataset:", DATASET)
print("shape:", frame.shape)
frame.head()


dataset: /Users/chathurangishyalika/IBM/AssetOpsBench/artifacts/kdd_tutorial/tsfm_tutorial_series.csv
shape: (120, 3)


,timestamp,temperature,pressure
0,2026-01-01 00:00:00,20.000000,100.500000
1,2026-01-01 01:00:00,20.141481,100.497502
2,2026-01-01 02:00:00,20.281017,100.490033
3,2026-01-01 03:00:00,20.416693,100.477668
4,2026-01-01 04:00:00,20.546652,100.460530


## 5. Profile the series

`profile_series` returns factual counts, channel statistics, and temporal evidence. It does not predict or diagnose.


In [7]:
profile_result = await call_tsfm(
    "profile_series",
    dataset_path=str(DATASET),
    timestamp_column="timestamp",
    channels=["temperature", "pressure"],
)
assert profile_result.get("n_observations") == N_OBSERVATIONS
profile_result


{'source': '/Users/chathurangishyalika/IBM/AssetOpsBench/artifacts/kdd_tutorial/tsfm_tutorial_series.csv',
 'n_observations': 120,
 'n_channels': 2,
 'dominant_period': 60,
 'channels': ['temperature', 'pressure'],
 'seasonality_strength': None,
 'trend_slope': None,
 'trend_strength': None,
 'non_stationary': False,
 'max_abs_channel_corr': None,
 'n_missing': 1,
 'value_range': [None, None]}

## 6. Characterize patterns

`characterize_series` produces shape-only evidence and an artifact pointer. It deliberately avoids assigning a fault label.


In [8]:
characterization = await call_tsfm(
    "characterize_series",
    dataset_path=str(DATASET),
    timestamp_column="timestamp",
    channels=["temperature", "pressure"],
    groups={"thermal": ["temperature"], "process": ["pressure"]},
)
assert characterization.get("status") == "success"
characterization


{'status': 'success',
 'summary': 'thermal: stable, near baseline; process: stable, near baseline.',
 'n_observations': 120,
 'evidence_file': 'file:///tmp/tsfm_work/pattern_evidence_57d0b805.json',
 'message': 'Pattern evidence (1 phase(s)). Full object at file:///tmp/tsfm_work/pattern_evidence_57d0b805.json.',
 'groups': {'thermal': ['temperature'], 'process': ['pressure']},
 'phases': [{'span': [0, 120],
   'per_group': {'thermal': {'state': 'STABLE',
     'rate': None,
     'magnitude': None,
     'persistence': None,
     'evidence': {'trend_r': None,
      'trend_strength': None,
      'half_diff': None,
      'range_norm': None,
      'concentration': None,
      'mean_cross_rate': 0.992,
      'autocorr1': 0.0,
      'flatline_fraction': 0.0,
      'max_abs_dev': None,
      'n_extreme': 0}},
    'process': {'state': 'STABLE',
     'rate': None,
     'magnitude': 1.06,
     'persistence': None,
     'evidence': {'trend_r': -0.137,
      'trend_strength': -0.31,
      'half_diff

## 7. Assess and clean data quality

The input CSV is preserved. The server writes a separate cleaned file and returns its pointer.


In [9]:
quality = await call_tsfm(
    "data_quality", dataset_path=str(DATASET), timestamp_column="timestamp"
)
assert quality.get("status") == "success"
assert quality.get("cleaned_file")
quality


{'status': 'success',
 'cleaned_file': 'file:///tmp/tsfm_work/cleaned_ba999d0e.csv',
 'rows_in': 120,
 'rows_out': 119,
 'message': 'Cleaned 120→119 rows. Cleaned series at file:///tmp/tsfm_work/cleaned_ba999d0e.csv.',
 'nan_per_column': {'timestamp': 0.0,
  'temperature': 0.8333333333333334,
  'pressure': 0.0},
 'removed_cost': 2}

## 8. Browse and shortlist models

These calls read the seeded `model_catalog` CouchDB collection. Model cards are metadata pointers, not checkpoint weights.


In [10]:
try:
    model_count = await call_tsfm("count_models")
    model_domains = await call_tsfm("list_domains", task_id="tsfm_forecasting")
    forecast_models = await call_tsfm(
        "find_models", task_id="tsfm_forecasting", top_k=5
    )
except RuntimeError as exc:
    raise RuntimeError(
        "TSFM catalogs are not ready. Complete the CouchDB setup from notebook 02 and load "
        "the default data with: uv run python src/couchdb/init_data.py"
    ) from exc

model_ids = [model["model_id"] for model in forecast_models.get("models", [])]
model_descriptions = (
    await call_tsfm("describe_models", model_ids=model_ids)
    if model_ids else {"models": [], "unknown": []}
)
{
    "count": model_count,
    "domains": model_domains,
    "shortlist": forecast_models,
    "descriptions": model_descriptions,
}


{'count': {'total': 1, 'by_task': {'tsfm_forecasting': 1}},
 'domains': {'domains': {'general': 1}},
 'shortlist': {'models': [{'_id': 'model:ttm_96_28',
    'model_id': 'ttm_96_28',
    'model_checkpoint': 'ttm_96_28',
    'model_family': 'TinyTimeMixer',
    'provenance': 'pretrained',
    'base_model_id': None,
    'task_ids': ['tsfm_forecasting'],
    'context_length': 96,
    'prediction_length': 28,
    'domain': 'general',
    'frequency': 'any',
    'source': 'local_artifact',
    'artifact_path': 'artifacts/tsfm_models/ttm_96_28',
    'hf_repo': 'ibm-granite/granite-timeseries-ttm-r2',
    'description': 'Pretrained TinyTimeMixer forecasting model, context length 96, prediction horizon 28. General-purpose zero-shot multivariate forecaster; good for short-context, short-horizon tasks.',
    'trained_on': ['pretraining-corpus'],
    'tags': ['ttm', 'forecasting', 'zero-shot', 'short-context'],
    'status': 'active',
    'version': 'r2',
    'created_by': 'seed',
    'created_at

## 9. Browse the feature catalog

Feature cards describe reusable extractors and transforms. This section reads metadata only; it does not register or update catalog entries.


In [11]:
feature_count = await call_tsfm("count_features")
features = await call_tsfm("list_features", status="active")
feature_ids = [
    feature.get("feature_id") for feature in features.get("features", [])
    if feature.get("feature_id")
][:5]
feature_descriptions = (
    await call_tsfm("describe_features", names=feature_ids)
    if feature_ids else {"features": [], "unknown": []}
)
{
    "count": feature_count,
    "catalog": features,
    "descriptions": feature_descriptions,
}


{'count': {'extractors': 1, 'transforms': 1, 'total': 2},
 'catalog': {'features': [{'_id': 'feature:abs_2nd_diff_mean',
    'feature_id': 'abs_2nd_diff_mean',
    'name': 'abs_2nd_diff_mean',
    'description': 'Mean absolute second difference.',
    'kind': 'extractor',
    'extractor_name': 'abs_2nd_diff_mean',
    'status': 'active',
    'dataset': 'feature_catalog'},
   {'_id': 'feature:efe_time_robust_norm_v1',
    'feature_id': 'efe_time_robust_norm_v1',
    'name': 'Invertible robust normalization',
    'modality': 'timeseries',
    'interface': 'fit_transform_inverse',
    'class_name': 'Transformation',
    'invertible': True,
    'code': 'import numpy as np\nclass Transformation:\n    """Invertible robust per-channel normalization (median/IQR) for forecasting inputs."""\n    def fit(self, X, metadata):\n        X = np.asarray(X, dtype=float)\n        center = np.median(X, axis=0)\n        q1, q3 = np.percentile(X, 25, axis=0), np.percentile(X, 75, axis=0)\n        scale = np

## 10. Inspect the recipe contract

The template is the authoritative description of forecasting, anomaly, ensemble, conformal, and tabular recipe shapes.


In [12]:
recipe_contract = await call_tsfm("recipe_template")
recipe_contract


{'task_choices': ['omitted / anything else  - FORECASTING: transforms -> single|ensemble -> conformal',
  'tsfm_anomaly_detection   - detector path, producing dense anomaly labels',
  'tsfm_classification | tsfm_regression | tsfm_clustering  - run_tabular_recipe only'],
 'estimator_spec': ['model_id     - a catalog card, e.g. {"model_id": "ttm_r1_512_96"}; its sktime_class and params are read from the card (see find_models / resolve_model)',
  'sktime_class - an inline class path + params, e.g. {"sktime_class": "sktime.forecasting.naive.NaiveForecaster", "params": {"strategy": "drift"}}'],
 'optional_blocks': ['fh         - forecast horizon, e.g. [1, 2, 3]. Default [1, 2, 3, 4, 5]',
  'transforms - list of transform specs applied to the target before the forecaster',
  'ensemble   - {"members": [<estimator spec>, ...], "combine": "mean|median|min|max|weighted|stack", "weights": [...]} - use INSTEAD of estimator',
  'conformal  - {"coverage": 0.9} for calibrated prediction intervals',
 

## 11. Optional lightweight forecasting recipe

This uses an inline `NaiveForecaster`, not a Torch checkpoint. Enable it by setting `KDD_RUN_TSFM_RECIPE=1` before starting Jupyter. The run writes output artifacts plus run/result ledger records; it does not change the input dataset or model/feature catalogs.


In [13]:
RUN_LIGHTWEIGHT_RECIPE = os.getenv("KDD_RUN_TSFM_RECIPE", "0") == "1"
RECIPE_TIMEOUT_SECONDS = int(os.getenv("KDD_TSFM_TIMEOUT_SECONDS", "180"))
DEMO_ASSET_ID = "kdd-tsfm-demo"
LIGHTWEIGHT_RECIPE = {
    "estimator": {
        "sktime_class": "sktime.forecasting.naive.NaiveForecaster",
        "params": {"strategy": "drift"},
    },
    "fh": [1, 2, 3, 4, 5],
    "eval": {"metrics": ["smape"]},
}

recipe_result = None
if RUN_LIGHTWEIGHT_RECIPE:
    try:
        recipe_result = await asyncio.wait_for(
            call_tsfm(
                "run_recipe",
                dataset_path=quality["cleaned_file"],
                timestamp_column="timestamp",
                target_columns=["temperature"],
                recipe=LIGHTWEIGHT_RECIPE,
                asset_id=DEMO_ASSET_ID,
            ),
            timeout=RECIPE_TIMEOUT_SECONDS,
        )
    except TimeoutError as exc:
        raise RuntimeError(
            f"TSFM recipe exceeded {RECIPE_TIMEOUT_SECONDS}s. Leave the optional run disabled "
            "or check the local ML dependencies."
        ) from exc
    display(recipe_result)
else:
    print("Recipe skipped. Set KDD_RUN_TSFM_RECIPE=1 before starting Jupyter to enable it.")


Recipe skipped. Set KDD_RUN_TSFM_RECIPE=1 before starting Jupyter to enable it.


## 12. Inspect run and result provenance

When execution is enabled, the returned `run_id` is resolved through `get_run`; `list_runs` and `list_results` demonstrate the persisted ledger.


In [14]:
if recipe_result:
    run_record = await call_tsfm("get_run", run_id=recipe_result["run_id"])
    runs = await call_tsfm("list_runs", asset_id=DEMO_ASSET_ID)
    result_listing = await call_tsfm(
        "list_results", task_type="tsfm_forecasting", asset_id=DEMO_ASSET_ID
    )
    result_rows = result_listing.get("results", [])
    result_record = None
    if result_rows:
        result_id = result_rows[0].get("result_id") or result_rows[0].get("_id")
        if result_id:
            result_record = await call_tsfm(
                "get_result", task_type="tsfm_forecasting", result_id=result_id
            )
    display({
        "run": run_record,
        "runs": runs,
        "results": result_listing,
        "selected_result": result_record,
    })
else:
    print("No recipe was executed, so no new run/result provenance is expected.")


No recipe was executed, so no new run/result provenance is expected.


## Safety and scope

The live TSFM server also exposes model/feature registration, update, deprecation, and versioning tools. They are intentionally not called because they mutate shared tutorial catalogs. `run_tabular_recipe`, `run_plan`, and multi-dataset `evaluate` are advanced execution paths better suited to dedicated examples with task-specific datasets.

## Takeaway

You exercised TSFM task discovery, file-pointer evidence, data quality, model and feature catalogs, recipe authoring, and optional lightweight execution with persisted provenance—without loading a heavyweight checkpoint or changing catalog metadata.
